# Etapa 4: Limpieza, Saneamiento Léxico y Gobernanza de Datos (SIPSA & EVA)

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA & EVA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 4: Data Preparation / Governance) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, SWEBOK, DAMA-BOK, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Homologación Léxica SIPSA**: Eliminación de caracteres especiales, estandarización de nombres de mercado y variedades.
2. **Gobernanza y Linaje EVA**: Auditoría de unicidad de clave compuesta `(cod_mpio, año, periodo_semestre, variedad)` y pureza de textos institucionales.
3. **Persistencia y Linaje Medallion**: Garantía de inmutabilidad en la capa `CLEANED` con formato Parquet optimizado.


In [5]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    fastparquet>=2023.8.0 \
    tabulate>=0.9.0

print(f"[OK] Ambiente verificado en Python: {sys.version.split()[0]}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
[OK] Ambiente verificado en Python: 3.13.5


In [6]:
# ==============================================================================
# CONFIGURACIÓN DE IMPORTS Y RUTAS DEL PROYECTO (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.infrastructure.config import CLEANED_DIR
from src.application.cleaning_service import CleaningService

print("Servicio de Limpieza importado exitosamente.")

Servicio de Limpieza importado exitosamente.


In [7]:
# ==============================================================================
# CARGA DE DATOS VALIDADOS
# ==============================================================================
parquet_path = CLEANED_DIR / "dataset_sipsa_mensual_nacional.parquet"

if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
else:
    # Fallback con variabilidad léxica
    df = pd.DataFrame({
        "fecha_mes": ["2023-01-01", "2023-01-01", "2023-01-01", "2023-01-01"],
        "año": [2023, 2023, 2023, 2023],
        "mes": [1, 1, 1, 1],
        "divipola_depto": ["05", "52", "25", "15"],
        "nombre_depto": ["ANTIOQUIA", "NARIÑO", "CUNDINAMARCA", "BOYACÁ"],
        "divipola_mpio": ["05400", "52399", "25875", "15861"],
        "nombre_mpio": ["LA UNION", "LA UNION", "VILLAPINZÓN", "VENTAQUEMADA"],
        "mercado_mayorista": ["MEDELLIN, CMA", "PASTO, EL POTRERILLO", "CORABASTOS", "CORABASTOS"],
        "alimento": ["PAPA", "PAPA", "PAPA", "PAPA"],
        "variedad_papa": ["PAPA DIACOL CAPIRO", "PAPA R-12 NEGRA", "PAPA SUPREMA", "PAPA CRIOLLA LIMPIA"],
        "volumen_ingreso_ton": [250.0, 180.0, 310.0, 95.0],
        "precio_prom_kg": [2300.0, 1950.0, 2400.0, 3100.0],
        "precio_min_kg": [2200.0, 1800.0, 2300.0, 2900.0],
        "precio_max_kg": [2450.0, 2100.0, 2550.0, 3300.0],
        "num_transacciones": [40, 35, 50, 20]
    })

print(f"Registros listos para saneamiento: {len(df):,}")
display(df.head())

Registros listos para saneamiento: 86,054


,fecha_mes,año,mes,divipola_depto,nombre_depto,divipola_mpio,nombre_mpio,mercado_mayorista,alimento,variedad_papa,volumen_ingreso_ton,precio_prom_kg,precio_min_kg,precio_max_kg,num_transacciones
0,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"CALI, CAVASA",PAPA,PAPA NEVADA,8.00,2187.94,2012.90,2406.73,1
1,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"MEDELLÍN, CENTRAL MAYORISTA DE ANTIOQUIA",PAPA,PAPA CRIOLLA,4.35,3263.66,3001.99,3590.18,12
2,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"MONTERÍA, MERCADO DEL SUR",PAPA,PAPA CAPIRA,105.55,2547.91,2338.02,2804.16,37
3,2019-01-01,2019,1,05,ANTIOQUIA,05001,MEDELLÍN,"MONTERÍA, MERCADO DEL SUR",PAPA,PAPA CRIOLLA,690.00,3220.37,2940.44,3570.77,12
4,2019-01-01,2019,1,05,ANTIOQUIA,05002,ABEJORRAL,"MEDELLÍN, CENTRAL MAYORISTA DE ANTIOQUIA",PAPA,PAPA CRIOLLA,1.80,3262.25,3001.27,3588.48,1


In [8]:
# ==============================================================================
# EJECUCIÓN DEL SERVICIO DE SANEAMIENTO Y DESAMBIGUACIÓN
# ==============================================================================
clean_service = CleaningService(df, output_dir=CLEANED_DIR)
df_limpio = clean_service.ejecutar_limpieza()

print("=== DISTRIBUCIÓN DE VARIEDADES CANÓNICAS HOMOLOGADAS ===")
print(df_limpio["variedad_papa"].value_counts())

print("\n=== VERIFICACIÓN DE DESAMBIGUACIÓN DE HOMÓNIMOS (LA UNIÓN) ===")
display(df_limpio[df_limpio["nombre_mpio"].str.contains("LA UNION")][["divipola_depto", "nombre_depto", "divipola_mpio", "nombre_mpio"]])

=== DISTRIBUCIÓN DE VARIEDADES CANÓNICAS HOMOLOGADAS ===
variedad_papa
PAPA OTRAS VARIEDADES    49139
PAPA CRIOLLA COLOMBIA    20183
PAPA PARDA PASTUSA        7389
PAPA PASTUSA SUPREMA      4990
PAPA RUBI                 2771
PAPA NEVADA               1582
Name: count, dtype: int64

=== VERIFICACIÓN DE DESAMBIGUACIÓN DE HOMÓNIMOS (LA UNIÓN) ===


,divipola_depto,nombre_depto,divipola_mpio,nombre_mpio
15,05,ANTIOQUIA,05400,LA UNION (ANT)
16,05,ANTIOQUIA,05400,LA UNION (ANT)
17,05,ANTIOQUIA,05400,LA UNION (ANT)
18,05,ANTIOQUIA,05400,LA UNION (ANT)
19,05,ANTIOQUIA,05400,LA UNION (ANT)
...,...,...,...,...
85030,05,ANTIOQUIA,05400,LA UNION (ANT)
85031,05,ANTIOQUIA,05400,LA UNION (ANT)
85032,05,ANTIOQUIA,05400,LA UNION (ANT)
85033,05,ANTIOQUIA,05400,LA UNION (ANT)


In [9]:
# ==============================================================================
# AUDITORÍA DE GOBERNANZA DATASET EVA: LINAJE, COMPLETITUD Y UNICIDAD
# ==============================================================================
import pandas as pd
from src.infrastructure.config import CLEANED_DIR

eva_path = CLEANED_DIR / "dataset_eva_agricola_nacional.parquet"
if eva_path.exists():
    df_eva = pd.read_parquet(eva_path)
    
    # Test de unicidad de clave municipal-semestral
    cols_clave = ["cod_mpio", "año", "periodo", "desagregacion_cultivo"]
    duplicados_clave = df_eva.duplicated(subset=cols_clave).sum()
    
    # Test de completitud por atributo DAMA-BOK
    nulos_por_col = df_eva.isna().sum()
    
    print("=== AUDITORÍA DE GOBERNANZA Y CALIDAD EVA (DAMA-BOK) ===")
    print(f"Filas totales: {len(df_eva):,}")
    print(f"Duplicados de clave compuesta {cols_clave}: {duplicados_clave}")
    print("\nCompletitud de atributos (% nulos):")
    print((nulos_por_col / len(df_eva) * 100).to_string())
    
    assert duplicados_clave == 0, "Existen duplicados en la clave de granularidad municipal EVA"
    assert nulos_por_col.max() == 0, "Existen valores nulos en atributos obligatorios EVA"
    print("\n✓ GOBERNANZA EVA CERTIFICADA: 100% de completitud y unicidad confirmadas.")


=== AUDITORÍA DE GOBERNANZA Y CALIDAD EVA (DAMA-BOK) ===
Filas totales: 5,574
Duplicados de clave compuesta ['cod_mpio', 'año', 'periodo', 'desagregacion_cultivo']: 0

Completitud de atributos (% nulos):
cod_depto                0.0
departamento             0.0
cod_mpio                 0.0
municipio                0.0
desagregacion_cultivo    0.0
cultivo                  0.0
ciclo_cultivo            0.0
grupo_cultivo            0.0
subgrupo                 0.0
año                      0.0
periodo                  0.0
area_sembrada_ha         0.0
area_cosechada_ha        0.0
produccion_ton           0.0
rendimiento_ton_ha       0.0
nombre_cientifico        0.0
cod_cultivo              0.0
estado_fisico            0.0

✓ GOBERNANZA EVA CERTIFICADA: 100% de completitud y unicidad confirmadas.


### Conclusiones de la Etapa 4:
- **SIPSA Limpio**: Los textos han sido purgados de caracteres especiales y acentos inconsistentes, estableciendo una clave de unión limpia con el catálogo DIVIPOLA.
- **EVA Gobernado**: El dataset oficial de producción agrícola cuenta con 100% de completitud, cero duplicados en su grano municipal-semestral y aislamiento formal de las dos variedades de papa.
- Los datos limpios se encuentran listos para el pipeline de **Ingeniería de Características (Features)**.
